# Scotland Weather Forecast Synthesiser — Ollama Edition

Same as `scotland_forecast.ipynb` but runs entirely **locally via Ollama** (llama3.2) — no API key needed, no data leaves your machine.

Covers:
- **Snow & Mountain conditions** (Highlands & Cairngorms)
- **Surf** (Thurso East)
- **Rain & General Weather**
- **Wind**

Run each cell top-to-bottom with `Shift+Enter`.

> **Note:** Make sure Ollama is running (`ollama serve`) and llama3.2 is pulled (`ollama pull llama3.2`) before running.

In [19]:
import requests
import requests as _requests
from IPython.display import Markdown, display
from openai import OpenAI
from playwright.async_api import async_playwright

In [20]:
try:
    r = requests.get("http://localhost:11434", timeout=3)
    print(r.text)
except Exception:
    print("Ollama is NOT running — open a terminal and run: ollama serve")

Ollama is running


In [21]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

## Scraper

Uses **Playwright** (headless Chromium) so JavaScript-rendered pages fully load before text extraction.

In [22]:
SCOTTISH_BUOYS = ['torness', 'arbroath', 'buckie', 'orkney', 'stonehaven', 'eyemouth']

def fetch_wavenet_buoys():
    try:
        r = _requests.get('https://wavenet-api.cefas.co.uk/api/Map/Current', timeout=10)
        features = r.json()['features']
        lines = ['WaveNet Live Scottish Buoy Readings\n']
        for f in features:
            title = f['properties']['title']
            if any(s in title.lower() for s in SCOTTISH_BUOYS):
                p = f['properties']
                results = p.get('results', {})
                lines.append(f'\n{title}  (as of {p["timestamp"]})')
                for v in results.values():
                    val = v['values'][0] if v['values'] else ''
                    if val:
                        lines.append(f'  {v["description"]}: {val} {v["unit"]}')
        return '\n'.join(lines)
    except Exception as e:
        return f'[Could not fetch WaveNet data: {e}]'


async def fetch_forecast(page, url, char_limit=5_000):
    try:
        await page.goto(url, wait_until='domcontentloaded', timeout=30_000)
        await page.wait_for_timeout(1500)
        for selector in ['button:has-text("AGREE")', 'button:has-text("Accept")',
                         'button:has-text("I agree")', 'button:has-text("Accept all")']:
            try:
                await page.click(selector, timeout=1500)
                await page.wait_for_timeout(800)
                break
            except:
                pass
        await page.evaluate("""
            document.querySelectorAll('script,style,nav,footer,header,img,input,iframe')
                     .forEach(el => el.remove())
        """)
        title = await page.title()
        body  = await page.inner_text('body')
        return (title + '\n\n' + body)[:char_limit]
    except Exception as e:
        return f'[Could not fetch {url}: {e}]'

In [23]:
BASE_SOURCES = {
    'Mountain & Snow — Cairngorms & Monadhliath (MWIS)':
        'https://www.mwis.org.uk/forecasts/scottish/cairngorms-np-and-monadhliath/text',
    'Mountain & Snow — West Highlands incl. Ben Nevis & Glencoe (MWIS)':
        'https://www.mwis.org.uk/forecasts/scottish/west-highlands/text',
    'Mountain & Snow — Northwest Highlands (MWIS)':
        'https://www.mwis.org.uk/forecasts/scottish/the-northwest-highlands/text',
    'Wind & Rain — Scotland (XCWeather)':
        'https://www.xcweather.co.uk/forecast/scotland',
    'General — Met Office Scotland':
        'https://www.metoffice.gov.uk/weather/forecast/gcpvj0v07',
}

In [24]:
SURF_SPOTS = {
    'Thurso East':      'https://www.surf-forecast.com/breaks/Thurso-East/forecasts/latest',
    'Pease Bay':        'https://www.surf-forecast.com/breaks/Pease-Bay/forecasts/latest',
    'Machrihanish':     'https://www.surf-forecast.com/breaks/Machrihanish/forecasts/latest',
    'Scarista':         'https://www.surf-forecast.com/breaks/Scarista/forecasts/latest',
    'Tiree (The Maze)': 'https://www.surf-forecast.com/breaks/The-Maze/forecasts/latest',
    'Coldingham Bay':   'https://www.surf-forecast.com/breaks/Coldingham-Bay/forecasts/latest',
    'Sandend Bay':      'https://www.surf-forecast.com/breaks/Sandend-Bay/forecasts/latest',
}

print('Available surf spots:')
for name in SURF_SPOTS:
    print(f'  - {name}')

Available surf spots:
  - Thurso East
  - Pease Bay
  - Machrihanish
  - Scarista
  - Tiree (The Maze)
  - Coldingham Bay
  - Sandend Bay


In [25]:
# Edit this list to choose which surf spots to include
selected_surf_spots = [
    'Thurso East',
    'Pease Bay',        # closest surf-forecast.com break to Belhaven Bay
    'Machrihanish',
]

FORECAST_SOURCES = dict(BASE_SOURCES)
for spot in selected_surf_spots:
    if spot in SURF_SPOTS:
        FORECAST_SOURCES[f'Surf — {spot}'] = SURF_SPOTS[spot]
    else:
        print(f"Warning: '{spot}' not found in SURF_SPOTS — check the name.")

print('Forecast sources for this run:')
for name in FORECAST_SOURCES:
    print(f'  - {name}')

Forecast sources for this run:
  - Mountain & Snow — Cairngorms & Monadhliath (MWIS)
  - Mountain & Snow — West Highlands incl. Ben Nevis & Glencoe (MWIS)
  - Mountain & Snow — Northwest Highlands (MWIS)
  - Wind & Rain — Scotland (XCWeather)
  - General — Met Office Scotland
  - Surf — Thurso East
  - Surf — Pease Bay
  - Surf — Machrihanish


## Fetch All Forecasts

Opens one headless Chromium browser, visits each source in turn, and extracts the rendered text. Expect this to take 20–40 seconds total.

In [26]:
raw_forecasts = {}

async with async_playwright() as p:
    browser = await p.chromium.launch(headless=True)
    page = await browser.new_page()

    for name, url in FORECAST_SOURCES.items():
        print(f'Fetching: {name}...')
        content = await fetch_forecast(page, url)
        raw_forecasts[name] = content
        print(f'  -> {len(content):,} characters retrieved\n')

    await browser.close()

print('Fetching: Live sea state — Scottish WaveNet buoys...')
raw_forecasts['Live Sea State — Scottish WaveNet Buoys'] = fetch_wavenet_buoys()
print(f'  -> {len(raw_forecasts["Live Sea State — Scottish WaveNet Buoys"]):,} characters retrieved\n')

Fetching: Mountain & Snow — Cairngorms & Monadhliath (MWIS)...
  -> 5,000 characters retrieved

Fetching: Mountain & Snow — West Highlands incl. Ben Nevis & Glencoe (MWIS)...
  -> 4,771 characters retrieved

Fetching: Mountain & Snow — Northwest Highlands (MWIS)...
  -> 5,000 characters retrieved

Fetching: Wind & Rain — Scotland (XCWeather)...
  -> 5,000 characters retrieved

Fetching: General — Met Office Scotland...
  -> 5,000 characters retrieved

Fetching: Surf — Thurso East...
  -> 5,000 characters retrieved

Fetching: Surf — Pease Bay...
  -> 5,000 characters retrieved

Fetching: Surf — Machrihanish...
  -> 5,000 characters retrieved

Fetching: Live sea state — Scottish WaveNet buoys...
  -> 1,387 characters retrieved



## Prompts

In [27]:
system_prompt = """
You are an expert Scottish weather analyst who knows the Highlands, the Cairngorms,
and the north coast surf scene well. You synthesise raw forecast data from multiple
sources into one clear, readable briefing.

Structure your response in markdown with these sections:
1. **Overview** - one-paragraph summary of the overall picture
2. **Snow & Mountain Conditions** - avalanche risk, snow depth, visibility, wind chill
3. **Surf** - swell height, period, wind direction, quality rating per spot
4. **Belhaven Bay / East Lothian Sea State** - use the Torness WaveNet buoy readings
5. **Rain & Wind** - precipitation amounts, wind speeds and directions, any warnings
6. **Outlook** - brief 3-5 day trend

If a section has no usable data, say so briefly rather than making things up.
Respond in markdown. Do not wrap the markdown in a code block.
"""

In [28]:
def build_user_prompt(forecasts):
    prompt = 'Please synthesise the following raw forecast data for Scotland into a structured briefing.\n\n'
    for source_name, content in forecasts.items():
        prompt += f'---\n### Source: {source_name}\n{content}\n\n'
    return prompt

## Synthesise!

Passes all fetched content to **llama3.2 running locally via Ollama** to produce the unified forecast briefing.

In [29]:
def synthesise_forecasts(forecasts):
    response = ollama.chat.completions.create(
        model='phi3',
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': build_user_prompt(forecasts)},
        ]
    )
    return response.choices[0].message.content

In [30]:
briefing = synthesise_forecasts(raw_forecasts)
display(Markdown(briefing))


| Date       | Location        | Significant Wave Height (m) | Temperature (°C) | Average (zero crossing) wave period (s) | Dominant (peak) wave period (s) | Wave spread (°) | Dominant (peak) wave direction (°) |
|------------|-----------------|----------------------------|------------------|---------------------------------------|------------------------------|-----------------|----------------------------------|
| 19/11/2026 | Arbroath SEPA    | 1.30                       | 10.00            | 4.3                                   | 7.7                          | 28              | 70                               |
| 19/11/2026 | Buckie SEPA      | 1.36                       | 10.80            | 5.0                                   | 8.7                          | 14              | 38                               |
| 12/08/2026 | Eyemouth SEPA    | N/A                        | 22.25            | N/A                                   | N/A                          | N/A             | N/A                              |
| 19/11/2026 | Orkney SEPA      | 0.65                       | 10.40            | 4.2                                   | 7.4                          | 31              | 101                              |
| 19/11/2term| Stonehaven SEPA  | 1.04                       | 10.30            | 4.8                                   | 6.9                          | 34              | 111                              |
| 19/11/2026 | Torness Power   | 1.49                       | 11.10            | 4.6                                   | 8.7                          | 23              | 39                               |


---

### Source: Live Sea State — Scottish WaveNet Buoys
WaveNet Live Scottish Buoy Readings


Arbroath SEPA waverider  (as of 2026-11-19T19:00:00)
  Significant Wave Height: 1.30 m
  Temperature: 10.00 °C
  Average (zero crossing) wave period: 4.3 s
  Dominant (peak) wave period: 7.7 s
  Wave spread: 28 °
  Dominant (peak) wave direction: 70 °

Buckie SEPA waverider  (as of 2026-11-19T19:00:00)
  Significant Wave Height: 1.36 m
  Temperature: 10.80 °C
  Average (zero crossing) wave period: 5.0 s
  Dominant (peak) wave period: 8.7 s
  Wave spread: 14 °
  Dominant (peak) wave direction: 38 °

Eyemouth SEPA Waverider  (as of 2026-08-12T07:00:00)
  Temperature: 22.25 °C

Orkney SEPA waverider  (as of 2026-11-19T19:00:00)
  Significant Wave Height: 0.65 m
  Temperature: 10.40 °C
  Average (zero crossing) wave period: 4.2 s
  Dominant (peak) wave period: 7.4 s
  Wave spread: 31 °
  Dominant (peak) wave direction: 101 °

Stonehaven SEPA waverider  (as of 2026-11-19T19:00:00)
  Significant Wave Height: 1.04 m
  Temperature: 10.30 °C
  Average (zero crossing) wave period: 4.8 s
  Dominant (peak) wave period: 6.9 s
  Wave spread: 34 °
  Dominant (peak) wave direction: 111 °

Torness Power Station Waverider  (as of 2026-11-19T19:00:00)
  Significant Wave Height: 1.49 m
  Temperature: 11.10 °C
  Average (zero crossing) wave period: 4.6 s
  Dominant (peak) wave period: 8.7 s
  Wave spread: 23 °
  Dominant (peak) wave direction: 39 °

